# Ensemble Analysis — Technique 2 (Member-wise Inverse-Variance Weighted)

This notebook introduces **Technique 2**, the most statistically rigorous ensemble method available in `quends`. Technique 2 computes a mean and variance **independently for each ensemble member**, then combines the per-member estimates using **inverse-variance weighting (IVW)**.

**Why inverse-variance weighting?**

IVW gives more influence to members with smaller variance (i.e., better-converged runs). A member that has run for a long time and has a tight confidence interval contributes more to the combined estimate than a short, noisy member. This is statistically optimal under the assumption that each member is an independent unbiased estimator of the same underlying quantity.

**Technique comparison overview:**

| Technique | Name | Handles length differences | Statistical rigor |
|-----------|------|---------------------------|-------------------|
| 0 | Average-then-analyze | No (long runs dominate) | Low |
| 1 | Pooled-block | Yes (block normalization) | Medium |
| **2** | **Member-wise IVW** | **Yes (variance-weighted)** | **High** |

## 1. Load and Trim the Ensemble

We load the same three GX simulation members used in Notebook 04 and apply ensemble trimming with `method="std"` to remove the transient startup phase from each member.

In [ ]:
import quends as qnds

files = [
    "gx/ensemble/tprim_2_5_a.out.csv",
    "gx/ensemble/tprim_2_5_b.out.csv",
    "gx/ensemble/tprim_2_5_c.out.csv",
]

members = [qnds.from_csv(f) for f in files]
ens = qnds.Ensemble(members)
trimmed_ens = ens.trim("HeatFlux_st", method="std", window_size=20)

print(f"Ensemble: {len(ens.data_streams)} members loaded")
for i, ds in enumerate(trimmed_ens.data_streams):
    print(f"  Member {i}: {len(ds.data)} time steps after trimming")

## 2. Compute Statistics — Technique 2 (Member-wise IVW)

With `technique=2`, `quends` performs the following steps internally:

1. Compute the mean and variance for **each individual member** using its stationary data.
2. Compute the **weight** for each member as $w_i = 1 / \sigma_i^2$ (inverse of its variance).
3. Combine: $\bar{\mu} = \sum_i w_i \mu_i \;/\; \sum_i w_i$.
4. Propagate uncertainty to form the combined confidence interval.

In [ ]:
results = trimmed_ens.compute_statistics("HeatFlux_st", technique=2)
print(results)

## 3. Inspect Individual Member Statistics

To understand how member-wise IVW works in practice, we examine each member's statistics individually. Members with lower variance (tighter spread) will receive higher weight in the combined estimate.

In [ ]:
print("Per-member statistics for HeatFlux_st:")
print("-" * 55)

for i, ds in enumerate(trimmed_ens.data_streams):
    mean = ds.mean()
    ci   = ds.confidence_interval()
    ess  = ds.effective_sample_size()
    print(f"  Member {i}:")
    print(f"    mean               = {mean:.4f}")
    print(f"    confidence interval = {ci}")
    print(f"    effective N         = {ess:.1f}")

### Interpreting Member Weights

The width of each member's confidence interval is inversely related to its statistical weight. A narrow CI means the member's data is well-converged and it will carry more influence in the IVW combination. A wide CI means the member is noisier and contributes less.

## 4. Compare All Three Techniques Side by Side

Finally, we run all three techniques on the same trimmed ensemble and compare the results. For a well-converged ensemble the means should be close, but the confidence intervals may differ, reflecting the different statistical assumptions each technique makes.

In [ ]:
technique_names = {
    0: "Average-then-analyze",
    1: "Pooled-block        ",
    2: "Member-wise IVW     ",
}

print(f"{'Technique':<6}  {'Name':<24}  {'Mean':>10}  {'Confidence Interval'}")
print("-" * 70)

for t in [0, 1, 2]:
    r = trimmed_ens.compute_statistics("HeatFlux_st", technique=t)
    print(f"T{t}      {technique_names[t]}  {r['mean']:>10.4f}  {r['confidence_interval']}")

## 5. Export Results with Exporter

The `qnds.Exporter` utility provides a convenient way to display results as a formatted dataframe for review or downstream processing.

In [ ]:
# Display the Technique 2 results using the Exporter
exporter = qnds.Exporter()
exporter.display_dataframe(results)

## Summary

- **Technique 2 (Member-wise IVW)** is the most statistically rigorous ensemble technique in `quends`.
- It treats each ensemble member as an independent estimator and weights them by the inverse of their variance.
- Well-converged members (narrow CI) automatically receive more influence in the combined result.
- Use it when your members have different run lengths, different noise levels, or when you want the most defensible UQ estimate.
- Access it via `trimmed_ens.compute_statistics(col, technique=2)`.

**Next:** See `06_aligned_vs_nonaligned_ensembles.ipynb` to learn how `quends` handles ensembles where members start at different times or use different time grids.